Data Cleaning and Exploratory Data Analysis

In [ ]:
#Load Essential Libaries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
#Read the Data
Data = pd.read_csv(r"/content/Bank_Transaction_Fraud_Detection.csv")

Quick Overview of Dataset

In [ ]:
#info
Data.info()

In [ ]:
# Frist five Rows of Data
Data.head()

In [ ]:
#Last Five Rows Of Data
Data.tail()

In [ ]:
#Shape of Data
Data.shape

Handling Duplicates & Missing values

In [ ]:
# Checking Duplicates
Data.duplicated().sum()

In [ ]:
# Checking Missing values
Data.isnull().sum()

By this we can say the dataset contains no missing values and no duplicate entries

In [ ]:
#Data Type Check
Data.dtypes

In [ ]:
from datetime import time
from pandas.core.resample import TimedeltaIndexResamplerGroupby
#Type Casting
Data['Transaction_Date'] = pd.to_datetime(Data['Transaction_Date'])
Data['Transaction_Time'] = pd.to_datetime(Data['Transaction_Time'])

In [ ]:
# Recheck the DataTypes
Data.dtypes

In [ ]:
#Statistics of dataset
Data.describe()

In [ ]:
# Separate Numerical & Categorical Columns
# Numerical columns
numerical_columns = Data.select_dtypes(include=np.number).columns.tolist()
# Categorical columns (only object or category types)
categorical_columns = Data.select_dtypes(include=["object", "category"]).columns.tolist()
Date_columns = Data.select_dtypes(include=["datetime64[ns]"]).columns.tolist()

In [ ]:
numerical_columns

In [ ]:
categorical_columns

In [ ]:
Date_columns

Handling Outliers

In [ ]:
#Outlier detection
plt.figure(figsize=(16,16))
Data.boxplot(grid=False)
plt.xticks()
plt.show()

In [ ]:
for col in ['Age','Transaction_Amount','Account_Balance']:
    Q1 = Data[col].quantile(0.25)
    Q3 = Data[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = Data[(Data[col] < lower) | (Data[col] > upper)]
    print(col, "Outliers:", len(outliers))

In [ ]:
# Final check
Data.info()

In [ ]:
# Export Clean Dataset
import os
output_file = "Bank Transaction Fraud Detection_Clean.csv"
Data.to_csv(output_file, index=False)
print("Final clean dataset saved at:")
print(os.path.abspath(output_file))

EDA

In [ ]:
#EDA
Data_EDA = Data.copy()

In [ ]:
#First Moment Business Decision
# mean
mean = Data_EDA[numerical_columns].mean()
print(mean)

In [ ]:
#median
median = Data_EDA[numerical_columns].median()
print(median)

In [ ]:
#mode
mode1 = Data_EDA[numerical_columns].mode()
print(mode1)

In [ ]:
#mode
mode2 = Data_EDA[categorical_columns].mode()
print(mode2)

In [ ]:
#Second Moment Business Decision
# Standard Deviation
Standard_Deviation = Data_EDA[numerical_columns].std()
print(Standard_Deviation)

In [ ]:
#  Variance
Variance = Data_EDA[numerical_columns].var()
print(Variance)

In [ ]:
#Range
range = Data_EDA[numerical_columns].max() - Data_EDA[numerical_columns].min()
print(range)

In [ ]:
#Third Moment Business Decision
# skewness
skewness = Data_EDA[numerical_columns].skew()
print(skewness)

In [ ]:
#Fourth Moment Business Decision
#kurtosis
kurtosis = Data_EDA[numerical_columns].kurtosis()
print(kurtosis)

In [ ]:
del range

In [ ]:
# Univariate Analysis - Histogram + KDE
import seaborn as sns
import matplotlib.pyplot as plt
sns.set(style="whitegrid")
n_cols = 2
n_rows = (len(numerical_columns) + 1) // 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, n_rows*4))
axes = axes.flatten()
for i, col in enumerate(numerical_columns):
    sns.histplot(Data_EDA[col], kde=True, ax=axes[i], color="skyblue", bins=30)
    axes[i].set_title(f"Distribution of {col}")
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()

In [ ]:
#Bivariate Analysis
#Numerical vs Target (Boxplots)#
for col in numerical_columns:
    if col != 'Is_Fraud':
        plt.figure(figsize=(6,4))
        sns.boxplot(x='Is_Fraud', y=col, data=Data_EDA)
        plt.title(f"{col} vs Fraud")
        plt.show()

In [ ]:
#Transaction Amount vs Fraud
sns.boxplot(x='Is_Fraud', y='Transaction_Amount', data=Data_EDA)
plt.title("Transaction Amount vs Fraud")
plt.show()

In [ ]:
#Multivariate Analysis
#Correlation
plt.figure(figsize=(10,6))
sns.heatmap(Data_EDA[numerical_columns].corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Matrix")
plt.show()

In [ ]:
#Pairplot
sns.pairplot(Data_EDA[numerical_columns], hue='Is_Fraud')
plt.show()

In [ ]:
#Fraud Distribution by Transaction Amount
sns.scatterplot(
    x='Transaction_Amount',
    y='Account_Balance',
    hue='Is_Fraud',
    data=Data_EDA
)
plt.title("Transaction Amount vs Account Balance")
plt.show()

ML

In [ ]:
#Feature Engineering and Preprocessing
#Date Features from Transaction_Date
Data_EDA['Transaction_Year'] = Data_EDA['Transaction_Date'].dt.year
Data_EDA['Transaction_Month'] = Data_EDA['Transaction_Date'].dt.month
Data_EDA['Transaction_Day'] = Data_EDA['Transaction_Date'].dt.day
Data_EDA['Transaction_DayOfWeek'] = Data_EDA['Transaction_Date'].dt.dayofweek

In [ ]:
#Time Features from Transaction_Time
Data_EDA['Transaction_Hour'] = Data_EDA['Transaction_Time'].dt.hour
Data_EDA['Transaction_Minute'] = Data_EDA['Transaction_Time'].dt.minute

In [ ]:
#Transaction Amount Ratio
Data_EDA['Amount_to_Balance_Ratio'] = Data_EDA['Transaction_Amount'] / Data_EDA['Account_Balance']

In [ ]:
# Remaining Balance After Transaction
Data_EDA['Remaining_Balance'] = Data_EDA['Account_Balance'] - Data_EDA['Transaction_Amount']

In [ ]:
# High Value Transaction Flag
Data_EDA['High_Value_Transaction'] = (
    Data_EDA['Transaction_Amount'] >
    Data_EDA['Transaction_Amount'].quantile(0.95)
).astype(int)

In [ ]:
# Night Transaction Flag
Data_EDA['Night_Transaction'] = (
    (Data_EDA['Transaction_Hour'] >= 0) &
    (Data_EDA['Transaction_Hour'] <= 6)
).astype(int)

In [ ]:
# Location Mismatch Feature
Data_EDA['Location_Mismatch'] = (
    Data_EDA['Transaction_Location'] != Data_EDA['City']
).astype(int)

In [ ]:
# Device Risk Flag
Data_EDA['New_Device'] = (
    Data_EDA['Device_Type'] == 'Unknown'
).astype(int)

In [ ]:
# Transaction Amount Category
Data_EDA['Amount_Category'] = pd.qcut(
    Data_EDA['Transaction_Amount'],
    q=4,
    labels=['Low','Medium','High','Very High']
)

In [ ]:
# Drop Unnecessary Columns
drop_cols = [
    'Customer_ID',
    'Customer_Name',
    'Customer_Email',
    'Customer_Contact',
    'Transaction_ID',
    'Transaction_Description',
    'Merchant_ID',
    'City',
    'Bank_Branch',
    'Transaction_Location'
]
Data_EDA = Data_EDA.drop(columns=drop_cols)
#They don't help prediction.

In [ ]:
Data_EDA.info()

In [ ]:
categorical_cols = [
    'Gender',
    'State',
    'Account_Type',
    'Transaction_Type',
    'Merchant_Category',
    'Transaction_Device',
    'Device_Type',
    'Transaction_Currency'
]

In [ ]:
Data_encoded = pd.get_dummies(Data_EDA, columns=categorical_cols, drop_first=True)

In [ ]:
#Verify Encoded Data
Data_encoded.shape

In [ ]:
X = Data_encoded.drop('Is_Fraud', axis=1)
y = Data_encoded['Is_Fraud']

In [ ]:
#Train Test Split
from sklearn.model_selection import train_test_split

# Drop rows where 'Is_Fraud' (y) is NaN
# Find indices where 'Is_Fraud' is NaN
nan_indices = y[y.isna()].index

# Drop these rows from both X and y
X_cleaned = X.drop(nan_indices)
y_cleaned = y.drop(nan_indices)

X_train, X_test, y_train, y_test = train_test_split(
    X_cleaned, # Use cleaned X
    y_cleaned, # Use cleaned y
    test_size=0.2,
    random_state=42,
    stratify=y_cleaned # Stratify using cleaned y
)

In [ ]:
#Check Split
print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

In [ ]:
#Check Class Distribution
print(y_train.value_counts())
print(y_test.value_counts())

In [ ]:
X_train = X_train.drop(['Transaction_Date','Transaction_Time'], axis=1, errors='ignore')
X_test = X_test.drop(['Transaction_Date','Transaction_Time'], axis=1, errors='ignore')

In [ ]:
X_train.dtypes

In [ ]:
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

In [ ]:
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

In [ ]:
# Handle Class Imbalance
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

In [ ]:
#Feature Scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_smote)
X_test_scaled = scaler.transform(X_test)

Modeling

In [ ]:
#Decision Tree
from sklearn.tree import DecisionTreeClassifier
DT = DecisionTreeClassifier(random_state=42)
DT.fit(X_train_scaled, y_train_smote)
y_pred_DT = DT.predict(X_test_scaled)

In [ ]:
#Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier
RF = RandomForestClassifier(random_state=42)
RF.fit(X_train_scaled, y_train_smote)
y_pred_RF = RF.predict(X_test_scaled)

In [ ]:
#Logistic Regression
from sklearn.linear_model import LogisticRegression
LR = LogisticRegression(max_iter=1000)
LR.fit(X_train_scaled, y_train_smote)
y_pred_LR = LR.predict(X_test_scaled)

In [ ]:
# Gradient Boosting
from sklearn.ensemble import GradientBoostingClassifier
GB = GradientBoostingClassifier(random_state=42)
GB.fit(X_train_scaled, y_train_smote)
y_pred_GB = GB.predict(X_test_scaled)

In [ ]:
#Evaluation Metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

def evaluate_model(y_true, y_pred):

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    return {
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1 Score": f1,
        "Confusion Matrix": cm
    }

In [ ]:
# Evaluate All Models
results = pd.DataFrame({
    "Logistic Regression": evaluate_model(y_test, y_pred_LR),
    "Decision Tree": evaluate_model(y_test, y_pred_DT),
    "Random Forest": evaluate_model(y_test, y_pred_RF),
    "Gradient Boosting": evaluate_model(y_test, y_pred_GB)
})

In [ ]:
#Display Results
print(results)

In [ ]:
#Confusion Matrix
import seaborn as sns
import matplotlib.pyplot as plt
cm = confusion_matrix(y_test, y_pred_DT)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix - Decision Tree")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
cm = confusion_matrix(y_test, y_pred_RF)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix - Random Forest")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
cm = confusion_matrix(y_test, y_pred_LR)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix - Logistic Regression")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
cm = confusion_matrix(y_test, y_pred_GB)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix - Gradient Boosting")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
# Model Comparison Table
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results = pd.DataFrame({
    "Model": ["Decision Tree", "Random Forest", "Logistic Regression", "Gradient Boosting"],

    "Accuracy": [
        accuracy_score(y_test, y_pred_DT),
        accuracy_score(y_test, y_pred_RF),
        accuracy_score(y_test, y_pred_LR),
        accuracy_score(y_test, y_pred_GB)
    ],

    "Precision": [
        precision_score(y_test, y_pred_DT, zero_division=0),
        precision_score(y_test, y_pred_RF, zero_division=0),
        precision_score(y_test, y_pred_LR, zero_division=0),
        precision_score(y_test, y_pred_GB, zero_division=0)
    ],

    "Recall": [
        recall_score(y_test, y_pred_DT, zero_division=0),
        recall_score(y_test, y_pred_RF, zero_division=0),
        recall_score(y_test, y_pred_LR, zero_division=0),
        recall_score(y_test, y_pred_GB, zero_division=0)
    ],

    "F1 Score": [
        f1_score(y_test, y_pred_DT, zero_division=0),
        f1_score(y_test, y_pred_RF, zero_division=0),
        f1_score(y_test, y_pred_LR, zero_division=0),
        f1_score(y_test, y_pred_GB, zero_division=0)
    ]
})

results

In [ ]:
# Comparision of Evaluation metrics
import seaborn as sns
import matplotlib.pyplot as plt
results_melted = results.melt(id_vars="Model", var_name="Metric", value_name="Score")
plt.figure(figsize=(10,6))
sns.barplot(data=results_melted, x="Model", y="Score", hue="Metric")
plt.title("Machine Learning Model Comparison")
plt.ylabel("Score")
plt.xticks(rotation=20)
plt.legend()
plt.show()

**Conclusion**

In this project, multiple machine learning models—Decision Tree, Random Forest, Logistic Regression, and Gradient Boosting—were developed to detect fraudulent transactions using financial transaction data. The dataset underwent comprehensive preprocessing, including data cleaning, exploratory data analysis, feature engineering, categorical encoding, handling class imbalance using SMOTE, and feature scaling to ensure data quality and model readiness.

Model performance was evaluated using Accuracy, Precision, Recall, and F1 Score. While Logistic Regression, Random Forest, and Gradient Boosting achieved high accuracy values (approximately 94–95%), accuracy alone proved to be misleading due to the highly imbalanced nature of the dataset. These models largely predicted transactions as legitimate, resulting in extremely low recall and F1 scores, indicating poor fraud detection capability.

The Decision Tree model, although having lower accuracy (~87.6%), demonstrated comparatively better performance in detecting fraudulent transactions, achieving a recall of approximately 6.59% and an F1 score of 5.08%. This highlights an important insight: models with slightly lower accuracy may perform better in identifying minority class instances such as fraud.

Overall, the results emphasize that fraud detection is a challenging problem due to severe class imbalance. Future improvements should focus on techniques such as threshold tuning, advanced ensemble methods (e.g., XGBoost), improved feature engineering, and anomaly detection approaches to enhance the model’s ability to accurately identify fraudulent transactions.